In [3]:
import subprocess
import re
import shutil
from pathlib import Path


# ============================================================
# USER SETTINGS
# ============================================================

# Vienna Philharmonic reference audio
REFERENCE_AUDIO = Path(
    r"C:\Users\stagk\Desktop\viennaPhilharmonic2026\viennaPhilharmonic2026_30min.mp3"
)

# Original video acquired at 50 fps
VIDEO_FILE = Path(
    r"C:\Users\stagk\Desktop\mouse1055494_Day01_mouseView_comp.avi"
)

# External mouse audio acquired at 300 kHz
MOUSE_AUDIO = Path(
    r"C:\Users\stagk\Desktop\20260825_mouse1055494_obs1.wav"
)

# Output file
#
# AVI is used because the original video is already in AVI format.
# The compressed video stream will be copied directly without re-encoding.
OUTPUT_FILE = Path(
    r"C:\Users\stagk\Desktop\mouse1055494_Day01_mouseView_with_audio.avi"
)

# Full path to FFmpeg
FFMPEG = Path(
    r"C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe"
)

# Mouse audio sampling rate
AUDIO_SAMPLE_RATE = 300000


# ============================================================
# CHECK INPUTS
# ============================================================

def check_files():
    """Check that all required files exist."""

    print("=" * 75)
    print("CHECKING INPUT FILES")
    print("=" * 75)

    files = {
        "Reference audio": REFERENCE_AUDIO,
        "Video": VIDEO_FILE,
        "Mouse audio": MOUSE_AUDIO,
        "FFmpeg": FFMPEG,
    }

    for name, path in files.items():

        if not path.exists():
            raise FileNotFoundError(
                f"\n{name} could not be found:\n"
                f"{path}\n"
            )

        print(f"{name:20s}: OK")
        print(f"    {path}")

    print()


# ============================================================
# RUN FFMPEG
# ============================================================

def run_ffmpeg(command, description="FFmpeg command"):
    """
    Run FFmpeg and capture its output.

    If FFmpeg fails, print the complete error rather than returning
    only a CalledProcessError.
    """

    print("\n" + "=" * 75)
    print(description)
    print("=" * 75)

    print("\nCommand:\n")
    print(
        " ".join(
            f'"{str(x)}"' if " " in str(x) else str(x)
            for x in command
        )
    )
    print()

    result = subprocess.run(
        [str(x) for x in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:

        print("\n" + "=" * 75)
        print("FFMPEG ERROR")
        print("=" * 75)
        print(result.stderr)

        raise RuntimeError(
            f"\nFFmpeg failed with return code "
            f"{result.returncode}.\n"
            f"See the FFmpeg output above for details."
        )

    return result


# ============================================================
# MEASURE AUDIO VOLUME
# ============================================================

def get_volume_stats(audio_file):
    """
    Measure mean and maximum volume using FFmpeg's volumedetect.

    Returns
    -------
    mean_volume : float
        Average RMS-like level in dBFS.

    max_volume : float
        Maximum detected level in dBFS.
    """

    command = [
        FFMPEG,
        "-hide_banner",
        "-nostats",
        "-i", audio_file,
        "-vn",
        "-af", "volumedetect",
        "-f", "null",
        "-"
    ]

    result = subprocess.run(
        [str(x) for x in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:

        print(result.stderr)

        raise RuntimeError(
            f"FFmpeg could not analyze:\n{audio_file}"
        )

    # --------------------------------------------------------
    # Find mean volume
    # --------------------------------------------------------

    mean_match = re.search(
        r"mean_volume:\s*(-?(?:\d+(?:\.\d+)?|inf))\s*dB",
        result.stderr
    )

    # --------------------------------------------------------
    # Find maximum volume
    # --------------------------------------------------------

    max_match = re.search(
        r"max_volume:\s*(-?(?:\d+(?:\.\d+)?|inf))\s*dB",
        result.stderr
    )

    if mean_match is None:
        raise RuntimeError(
            f"\nCould not determine mean volume for:\n"
            f"{audio_file}\n\n"
            f"FFmpeg output:\n"
            f"{result.stderr}"
        )

    if max_match is None:
        raise RuntimeError(
            f"\nCould not determine maximum volume for:\n"
            f"{audio_file}\n"
        )

    mean_string = mean_match.group(1)
    max_string = max_match.group(1)

    if mean_string == "-inf":
        raise RuntimeError(
            f"\nThe audio appears to contain only silence:\n"
            f"{audio_file}"
        )

    mean_volume = float(mean_string)

    if max_string == "-inf":
        max_volume = float("-inf")
    else:
        max_volume = float(max_string)

    return mean_volume, max_volume


# ============================================================
# FUSE VIDEO + AUDIO
# ============================================================

def fuse_video_audio(
    video_file,
    audio_file,
    output_file,
    gain_db
):
    """
    Add the external mouse audio to the video.

    IMPORTANT
    ---------
    - Video is copied directly with -c:v copy.
    - There is NO video re-encoding.
    - Therefore there is NO additional video compression.
    - Original video frames remain encoded exactly as they were.
    - Only the audio is processed.
    """

    command = [
        FFMPEG,

        # Overwrite output
        "-y",

        # ----------------------------------------------------
        # INPUT 0 = video
        # ----------------------------------------------------
        "-i", video_file,

        # ----------------------------------------------------
        # INPUT 1 = external 300-kHz mouse audio
        # ----------------------------------------------------
        "-i", audio_file,

        # ----------------------------------------------------
        # STREAM SELECTION
        # ----------------------------------------------------

        # Take ONLY video from input 0
        "-map", "0:v:0",

        # Take ONLY audio from input 1
        "-map", "1:a:0",

        # ----------------------------------------------------
        # VIDEO
        # ----------------------------------------------------
        #
        # CRITICAL:
        #
        # Stream-copy the already-compressed video.
        #
        # No decoding/re-encoding.
        # No additional compression.
        # No quality loss.
        #
        "-c:v", "copy",

        # ----------------------------------------------------
        # AUDIO
        # ----------------------------------------------------

        # Adjust volume to match the Philharmonic mean level
        "-af", f"volume={gain_db:.6f}dB",

        # Use 32-bit floating-point PCM.
        #
        # This is uncompressed audio and is useful here because
        # a large positive volume adjustment can exceed the
        # normal integer PCM range.
        "-c:a", "pcm_f32le",

        # Explicitly preserve the 300-kHz sampling rate
        "-ar", str(AUDIO_SAMPLE_RATE),

        # Stop when the shorter input finishes
        "-shortest",

        # Output
        output_file
    ]

    run_ffmpeg(
        command,
        description="CREATING VIDEO WITH MATCHED AUDIO"
    )


# ============================================================
# MAIN
# ============================================================

def main():

    print()
    print("=" * 75)
    print("MOUSE VIDEO + 300 kHz AUDIO")
    print("MATCH AUDIO LEVEL TO VIENNA PHILHARMONIC")
    print("=" * 75)
    print()

    # --------------------------------------------------------
    # 1. Check that everything exists
    # --------------------------------------------------------

    check_files()

    # --------------------------------------------------------
    # 2. Measure reference volume
    # --------------------------------------------------------

    print("=" * 75)
    print("MEASURING VIENNA PHILHARMONIC")
    print("=" * 75)

    reference_mean, reference_max = get_volume_stats(
        REFERENCE_AUDIO
    )

    print(
        f"\nPhilharmonic mean volume : "
        f"{reference_mean:.2f} dBFS"
    )

    print(
        f"Philharmonic max volume  : "
        f"{reference_max:.2f} dBFS"
    )

    # --------------------------------------------------------
    # 3. Measure mouse recording
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("MEASURING MOUSE AUDIO")
    print("=" * 75)

    mouse_mean, mouse_max = get_volume_stats(
        MOUSE_AUDIO
    )

    print(
        f"\nMouse mean volume        : "
        f"{mouse_mean:.2f} dBFS"
    )

    print(
        f"Mouse max volume         : "
        f"{mouse_max:.2f} dBFS"
    )

    # --------------------------------------------------------
    # 4. Calculate required gain
    # --------------------------------------------------------

    gain_db = reference_mean - mouse_mean

    predicted_max = mouse_max + gain_db

    print()
    print("=" * 75)
    print("VOLUME MATCHING")
    print("=" * 75)

    print(
        f"\nReference mean : {reference_mean:.2f} dBFS"
    )

    print(
        f"Mouse mean     : {mouse_mean:.2f} dBFS"
    )

    print(
        f"Required gain  : {gain_db:+.2f} dB"
    )

    print(
        f"Mouse peak before gain : {mouse_max:.2f} dBFS"
    )

    print(
        f"Predicted peak after gain: "
        f"{predicted_max:.2f} dBFS"
    )

    # --------------------------------------------------------
    # Warn about possible >0 dBFS values
    # --------------------------------------------------------

    if predicted_max > 0:

        print()
        print("WARNING:")
        print(
            "The requested gain makes some samples exceed "
            "0 dBFS."
        )

        print(
            "The output therefore uses 32-bit floating-point "
            "PCM so FFmpeg does not need to clip those samples "
            "during storage."
        )

        print(
            "No compressor or limiter will be applied because "
            "that would alter the waveform."
        )

    # --------------------------------------------------------
    # 5. Create output
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("OUTPUT SETTINGS")
    print("=" * 75)

    print()
    print("Video:")
    print("  Codec        : COPY ORIGINAL STREAM")
    print("  Re-encoding  : NO")
    print("  Compression  : NO ADDITIONAL COMPRESSION")
    print("  Frame rate   : unchanged from source")

    print()
    print("Audio:")
    print(f"  Sample rate  : {AUDIO_SAMPLE_RATE} Hz")
    print("  Codec        : 32-bit floating-point PCM")
    print("  Compression  : none")
    print(f"  Gain         : {gain_db:+.2f} dB")

    fuse_video_audio(
        VIDEO_FILE,
        MOUSE_AUDIO,
        OUTPUT_FILE,
        gain_db
    )

    # --------------------------------------------------------
    # 6. Verify output audio
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("VERIFYING OUTPUT AUDIO")
    print("=" * 75)

    output_mean, output_max = get_volume_stats(
        OUTPUT_FILE
    )

    difference = output_mean - reference_mean

    # --------------------------------------------------------
    # 7. Final report
    # --------------------------------------------------------

    print()
    print("=" * 75)
    print("FINISHED")
    print("=" * 75)

    print()
    print(
        f"Philharmonic mean : {reference_mean:.2f} dBFS"
    )

    print(
        f"Original mouse    : {mouse_mean:.2f} dBFS"
    )

    print(
        f"Applied gain      : {gain_db:+.2f} dB"
    )

    print(
        f"Output mean       : {output_mean:.2f} dBFS"
    )

    print(
        f"Difference        : {difference:+.2f} dB"
    )

    print(
        f"Output max        : {output_max:.2f} dBFS"
    )

    print()
    print("Output file:")
    print(OUTPUT_FILE)

    print()
    print(
        "The video stream was copied without re-encoding."
    )

    print(
        "Therefore the already-compressed video was NOT "
        "compressed a second time."
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()


MOUSE VIDEO + 300 kHz AUDIO
MATCH AUDIO LEVEL TO VIENNA PHILHARMONIC

CHECKING INPUT FILES
Reference audio     : OK
    C:\Users\stagk\Desktop\viennaPhilharmonic2026\viennaPhilharmonic2026_30min.mp3
Video               : OK
    C:\Users\stagk\Desktop\mouse1055494_Day01_mouseView_comp.avi
Mouse audio         : OK
    C:\Users\stagk\Desktop\20260825_mouse1055494_obs1.wav
FFmpeg              : OK
    C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe

MEASURING VIENNA PHILHARMONIC

Philharmonic mean volume : -16.80 dBFS
Philharmonic max volume  : 0.00 dBFS

MEASURING MOUSE AUDIO

Mouse mean volume        : -42.70 dBFS
Mouse max volume         : -0.00 dBFS

VOLUME MATCHING

Reference mean : -16.80 dBFS
Mouse mean     : -42.70 dBFS
Required gain  : +25.90 dB
Mouse peak before gain : -0.00 dBFS
Predicted peak after gain: 25.90 dBFS

The requested gain makes some samples exceed 0 dBFS.
The output therefore uses 32-bit floating-point PCM so FFmpeg does not need to clip those samples during storage.